# Lab 1 — Stand Up and Compare
**AI Engineering for Managers · Day 1 · 09:55–10:45**

**Goal:** prove your environment works, call two very different models on the same task, and walk away with numbers — latency, tokens, cost — not impressions.

**How to work:** run every cell top to bottom (`Run All` works). Cells marked **✏️ YOUR TURN** are where you change things.

> Rule for the day: *never say "the model is good/bad" without a number next to it.*

In [ ]:
# SETUP — run this first. It is the same in every lab.
import os, time, json

from openai import OpenAI, AzureOpenAI

ENDPOINT       = os.environ.get("FOUNDRY_ENDPOINT", "")
API_KEY        = os.environ.get("FOUNDRY_API_KEY", "")
MODEL_MINI     = os.environ.get("MODEL_MINI", "gpt-4o-mini")          # small, cheap workhorse
MODEL_FRONTIER = os.environ.get("MODEL_FRONTIER", "gpt-4o")           # large, expensive
MODEL_EMBED    = os.environ.get("MODEL_EMBED", "text-embedding-3-small")

assert ENDPOINT and API_KEY, (
    "Model endpoint not configured. On the lab VMs these are pre-set — "
    "if you see this, call the trainer."
)

# Works with both Azure OpenAI endpoints and OpenAI-compatible serverless endpoints
if any(h in ENDPOINT for h in ("openai.azure.com", "cognitiveservices", "services.ai.azure.com")):
    client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=API_KEY, api_version="2024-10-21")
else:
    client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

print("Client ready.")
print("Mini model     :", MODEL_MINI)
print("Frontier model :", MODEL_FRONTIER)

In [ ]:
# Helpers used all day. with_retry keeps the room moving if the endpoint rate-limits briefly.
def with_retry(fn, tries=3, wait=5):
    for a in range(tries):
        try:
            return fn()
        except Exception as e:
            # 400 = bad request: retrying won't help, fail fast with the real message
            if a == tries - 1 or getattr(e, "status_code", None) == 400: raise
            print(f"  (retrying in {wait*(a+1)}s: {e})")
            time.sleep(wait * (a + 1))

def ask(model, messages, temperature=0.2, **kw):
    """Call a chat model and return text + latency + token counts."""
    t0 = time.perf_counter()
    params = dict(model=model, messages=messages, **kw)
    if model != MODEL_FRONTIER:          # frontier reasoning model accepts only the default temperature
        params["temperature"] = temperature
    r = with_retry(lambda: client.chat.completions.create(**params))
    latency = time.perf_counter() - t0
    u = r.usage
    return {
        "model": model,
        "text": r.choices[0].message.content,
        "latency_s": round(latency, 2),
        "input_tokens": u.prompt_tokens,
        "output_tokens": u.completion_tokens,
    }

# Prices in USD per 1 MILLION tokens.
# >>> EDIT these to match the actual prices of your deployments. <<<
PRICE = {
    MODEL_MINI:     {"in": 0.75, "out": 4.50},
    MODEL_FRONTIER: {"in": 5.50, "out": 33.00},
}

def cost_usd(result):
    p = PRICE[result["model"]]
    return round((result["input_tokens"] * p["in"] +
                  result["output_tokens"] * p["out"]) / 1_000_000, 6)

## 1.1 — First call
One call to the mini model. If this works, your whole environment works.

In [ ]:
r = ask(MODEL_MINI, [{"role": "user", "content": "In one sentence: what does a variable-frequency drive do?"}])
print(r["text"])
print(f'\n{r["latency_s"]}s · {r["input_tokens"]} in / {r["output_tokens"]} out · ${cost_usd(r)}')

## 1.2 — Same task, two models
A realistic task from our case company, **Axion Controls**: a service engineer pastes a messy field report and wants a structured diagnosis. We give the *identical* prompt to the mini model and the frontier model.

In [ ]:
FIELD_REPORT = """Site: Everpack Foods, Coimbatore. Drive on the packing line trips overtemp
F-07 about every hour. Filters are clean, fan spins at power-up. It's set to 10 kHz switching
because of noise near the office. Firmware says 2.3. Cabinet gets warm in the afternoon,
maybe 38 degrees. Customer wants to order a new fan kit — should we?"""

TASK = [{"role": "system",
         "content": "You are a service-desk assistant for Axion Controls industrial products. "
                    "Be concise. Structure your answer as: Likely cause / Checks / Recommended action."},
        {"role": "user", "content": FIELD_REPORT}]

r_mini     = ask(MODEL_MINI, TASK)
r_frontier = ask(MODEL_FRONTIER, TASK)

print("=== MINI ===\n",     r_mini["text"])
print("\n=== FRONTIER ===\n", r_frontier["text"])

## 1.3 — The comparison sheet
Same prompt, two models. Now the numbers.

In [ ]:
import pandas as pd

rows = []
for r in (r_mini, r_frontier):
    rows.append({"model": r["model"], "latency_s": r["latency_s"],
                 "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                 "cost_usd": cost_usd(r),
                 "cost_per_1000_queries_usd": round(cost_usd(r) * 1000, 2)})
df = pd.DataFrame(rows)
df

**Read that last column again.** That is the difference per 1,000 service-desk queries for *one task design*. Multiply by your team's real volume.

✏️ **YOUR TURN — fill in on paper or in the cell below (this is your Model Comparison Sheet):**
1. Did the frontier model's answer *decide* anything the mini model's did not?
2. The correct diagnosis here involves firmware 2.3 + 10 kHz switching (it's a known bulletin pattern — you'll meet it again in Lab 3). Did either model get there *without* being given the manuals? What does that tell you?
3. At what price ratio would you route this task to the mini model?

In [ ]:
MY_NOTES = """
1. Decision difference:
2. Neither model knows OUR products without context because:
3. My routing rule for this task:
"""
print(MY_NOTES)

## 1.4 — Stretch (if you have time)
Change `temperature` to 0.9 in a call above and re-run it twice. Watch the answers change. Then ask: *which of my team's use cases can tolerate that?*

## 1.5 — Result line
Run this cell, copy the printed line, and paste it into the **Paste & check** box for Lab 1 on the course page.

In [ ]:
print(f"LAB1|mini_per_1k={cost_usd(r_mini)*1000:.2f}|frontier_per_1k={cost_usd(r_frontier)*1000:.2f}")

✅ **Done.** You have: a working environment, and a model comparison sheet with real numbers. Keep this notebook open — Lab 2 builds on the same helpers.